# 05 Finalize Training Descriptions

This notebook writes the main dataset used for BERTopic/recommender training.

It combines:

1. exact source-grounded Plot/Synopsis/Premise extracts when available
2. batch Wikipedia full extracts when available
3. a no-hallucination metadata expansion for every remaining title

The metadata expansion does **not** invent plot facts. It only restates verified fields already in the dataset: title, year, type, genres, rating/vote context, and the current description.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_columns", 80)

DATA_DIR = Path("data")
BASE_DATA_PATH = DATA_DIR / "cleaned_watch_data.csv"
LONG_PLOTS_PATH = DATA_DIR / "extensive_long_plot_summaries.csv"
BATCH_EXTRACTS_PATH = DATA_DIR / "extensive_wikipedia_full_extract_fallbacks.csv"
FINAL_ENRICHED_PATH = DATA_DIR / "cleaned_watch_data_extensive_enriched.csv"
BERTOPIC_COMPAT_PATH = DATA_DIR / "cleaned_watch_data_phase2_enriched.csv"

MAX_SOURCE_WORDS = 900
MIN_GOOD_MODELING_WORDS = 140


In [2]:
def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def truncate_words(text, max_words):
    words = str(text).split()
    return " ".join(words[:max_words]) if len(words) > max_words else str(text)


def split_genres(value):
    return [item.strip() for item in str(value).split(",") if item.strip()]

GENRE_NOTES = {
    "Action": "action-driven conflict, physical stakes, danger, pursuit, combat, or urgent set pieces",
    "Adventure": "journey, discovery, external obstacles, movement across places, or quest-like progression",
    "Animation": "animated storytelling; this alone does not imply that the title is for children or families",
    "Biography": "life-story material connected to a real person, career, public identity, or historical figure",
    "Comedy": "humor, absurdity, wit, social awkwardness, comic situations, or lighter tonal movement",
    "Crime": "lawbreaking, investigation, criminals, justice systems, moral compromise, or consequences of crime",
    "Documentary": "nonfiction subject matter, real events, interviews, archival material, or explanatory structure",
    "Drama": "character conflict, emotional stakes, relationships, social pressure, or serious personal consequences",
    "Family": "family-accessible or family-centered storytelling according to the genre metadata",
    "Fantasy": "magic, myth, supernatural worldbuilding, impossible events, or invented realms",
    "History": "historical period, past events, political context, or real-world historical framing",
    "Horror": "fear, threat, dread, monsters, violence, disturbing events, or survival under terror",
    "Music": "musicians, performance, songs, musical culture, or music-industry context",
    "Musical": "storytelling that uses songs or musical performance as a major form",
    "Mystery": "unknown facts, secrets, investigation, clues, or delayed revelation",
    "News": "journalistic, current-affairs, or reported real-world material",
    "Reality-TV": "unscripted competition, lifestyle, observation, celebrity, or real-person format",
    "Romance": "love, desire, relationship formation, heartbreak, intimacy, or romantic conflict",
    "Sci-Fi": "science fiction, technology, speculative futures, artificial intelligence, space, or altered reality",
    "Short": "short-form storytelling or brief runtime format",
    "Sport": "athletic competition, training, teams, sports culture, or performance pressure",
    "Talk-Show": "conversation, interviews, commentary, guests, or host-driven format",
    "Thriller": "suspense, tension, danger, pursuit, conspiracy, or high-stakes uncertainty",
    "War": "military conflict, wartime society, soldiers, occupation, combat, or effects of war",
    "Western": "frontier settings, outlaws, lawmen, survival, revenge, or western genre conventions",
}

STOPWORDS = set("""
the a an and or but if then else for from to in on at of by with without into onto over under across through as is are was were be been being have has had do does did this that these those it its their his her him she he they them we us you your i me my mine our ours not no yes than so very just about after before during while where when who whom whose which what why how all any both each few more most other some such only own same too can will would should could may might must
title current description source grounded expanded metadata genre record movie series tv tvseries tvminiseries short video
""".split())

def description_keywords(description, n=12):
    tokens = re.findall(r"\b[a-zA-Z][a-zA-Z'-]{2,}\b", str(description).lower())
    tokens = [t.strip("'-") for t in tokens if t not in STOPWORDS]
    counts = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [token for token, _ in ranked[:n]]


def metadata_expansion(row):
    title = clean_text(row.get("primary_title", ""))
    display_title = clean_text(row.get("display_title", ""))
    original_title = clean_text(row.get("original_title", ""))
    content_type = clean_text(row.get("content_type", ""))
    year = clean_text(row.get("release_year", ""))
    genres = split_genres(row.get("genres", ""))
    description = clean_text(row.get("description", ""))
    rating = row.get("average_rating", np.nan)
    votes = row.get("num_votes", np.nan)
    weighted = row.get("weighted_rating", np.nan)
    keywords = description_keywords(description)

    genre_sentence = "; ".join(f"{g}: {GENRE_NOTES.get(g, 'genre metadata present in the source dataset')}" for g in genres)
    keyword_sentence = ", ".join(keywords) if keywords else "no strong description keywords available"

    animation_note = ""
    if "Animation" in genres:
        if "Family" in genres:
            animation_note = "The metadata marks this as animation with a Family genre, so the recommender may treat it as more family-accessible than adult animation, while still checking the actual description."
        elif any(g in genres for g in ["Action", "Horror", "Thriller", "Crime"]):
            animation_note = "The metadata marks this as animation combined with action, horror, thriller, or crime signals, so it should not be assumed to be children's or family animation."
        else:
            animation_note = "The metadata marks this as animation, but animation alone is not a maturity label; recommendations should use the full genre and description context."

    identity_bits = [title]
    if display_title and display_title != title:
        identity_bits.append(f"display title {display_title}")
    if original_title and original_title not in identity_bits:
        identity_bits.append(f"original title {original_title}")

    return " ".join(part for part in [
        f"Verified title identity: {'; '.join(identity_bits)}.",
        f"Verified format and release context: {content_type or 'unknown content type'} released in {year or 'unknown year'}.",
        f"Verified genre profile: {', '.join(genres) if genres else 'unknown genres'}.",
        f"Genre meaning for modeling and clustering: {genre_sentence}.",
        f"Current verified description: {description}",
        f"Description keywords present in the dataset text: {keyword_sentence}.",
        animation_note,
        f"Rating context for popularity and quality weighting: average rating {rating}, vote count {votes}, weighted rating {weighted}.",
        "No additional plot facts are invented in this generated expansion; it exists to give the NLP model clearer verified context when a full source-grounded plot is not available yet.",
    ] if part)


In [3]:
base = pd.read_csv(BASE_DATA_PATH).reset_index(drop=True)
base["row_id"] = np.arange(len(base))
base["description_word_count"] = base["description"].fillna("").map(word_count)

long_plots = pd.read_csv(LONG_PLOTS_PATH) if LONG_PLOTS_PATH.exists() and LONG_PLOTS_PATH.stat().st_size > 0 else pd.DataFrame(columns=["tconst"])
fallback = pd.read_csv(BATCH_EXTRACTS_PATH) if BATCH_EXTRACTS_PATH.exists() and BATCH_EXTRACTS_PATH.stat().st_size > 0 else pd.DataFrame(columns=["tconst"])

plot_cols = ["tconst", "wikidata_id", "wikidata_label", "wikipedia_title", "wikipedia_url", "source_section", "enrichment_text", "enrichment_word_count", "match_method", "license_note"]
plot_cols = [c for c in plot_cols if c in long_plots.columns]
plot_data = long_plots[plot_cols].drop_duplicates("tconst") if not long_plots.empty else long_plots

fallback_cols = ["tconst", "wikidata_id", "wikidata_label", "wikipedia_title", "wikipedia_url", "fallback_extract_text", "fallback_extract_word_count", "fallback_match_method", "license_note"]
fallback_cols = [c for c in fallback_cols if c in fallback.columns]
fallback_data = fallback[fallback_cols].drop_duplicates("tconst") if not fallback.empty else fallback

out = base.merge(plot_data, on="tconst", how="left")
out = out.merge(fallback_data, on="tconst", how="left", suffixes=("", "_fallback"))

for col in ["enrichment_text", "fallback_extract_text"]:
    if col not in out.columns:
        out[col] = ""

has_plot = out["enrichment_text"].fillna("").astype(str).str.strip().ne("")
has_fallback = out["fallback_extract_text"].fillna("").astype(str).str.strip().ne("")

out["source_enrichment_text"] = np.where(has_plot, out["enrichment_text"], out["fallback_extract_text"])
out["source_enrichment_type"] = np.select(
    [has_plot, has_fallback],
    ["plot_or_synopsis_section", "wikipedia_full_extract_fallback"],
    default="none",
)
out["metadata_expanded_description"] = out.apply(metadata_expansion, axis=1)
out["metadata_expanded_word_count"] = out["metadata_expanded_description"].map(word_count)

out["has_source_enrichment"] = has_plot | has_fallback
out["has_phase2_enrichment"] = out["has_source_enrichment"]
out["enrichment_text"] = np.where(out["has_source_enrichment"], out["source_enrichment_text"], "")
out["enrichment_word_count"] = out["enrichment_text"].fillna("").map(word_count)
out["final_enrichment_source_type"] = np.where(out["has_source_enrichment"], out["source_enrichment_type"], "original_description_only")


def final_modeling_description(row):
    original = clean_text(row.get("description", ""))
    source_text = clean_text(row.get("source_enrichment_text", ""))
    parts = []
    if original:
        parts.append(f"Current description: {original}")
    if source_text:
        parts.append(f"Source-grounded expanded text: {truncate_words(source_text, MAX_SOURCE_WORDS)}")
    return " ".join(parts)

out["modeling_description"] = out.apply(final_modeling_description, axis=1)

out["modeling_description_word_count"] = out["modeling_description"].map(word_count)
out["needs_manual_research"] = out["modeling_description_word_count"] < MIN_GOOD_MODELING_WORDS
out["description_rebuild_status"] = np.select(
    [has_plot, has_fallback],
    ["source_grounded_plot_or_synopsis_added", "source_grounded_full_extract_added"],
    default="original_description_only",
)

out.to_csv(FINAL_ENRICHED_PATH, index=False)
out.to_csv(BERTOPIC_COMPAT_PATH, index=False)

print("Saved:", FINAL_ENRICHED_PATH)
print("Saved:", BERTOPIC_COMPAT_PATH)
print("Rows:", len(out))
print("Rows with source enrichment:", int(out["has_source_enrichment"].sum()))
print("Rows covered with metadata expansion:", int((out["final_enrichment_source_type"] == "metadata_expanded_original_description").sum()))
print("Rows with phase2/modeling enrichment:", int(out["has_phase2_enrichment"].sum()))
print("Minimum modeling words:", int(out["modeling_description_word_count"].min()))
print("Median modeling words:", float(out["modeling_description_word_count"].median()))
print(out["final_enrichment_source_type"].value_counts().to_string())

display(out[["tconst", "primary_title", "release_year", "genres", "final_enrichment_source_type", "enrichment_word_count", "modeling_description_word_count"]].head(20))


Saved: data/cleaned_watch_data_extensive_enriched.csv
Saved: data/cleaned_watch_data_phase2_enriched.csv
Rows: 7848
Rows with source enrichment: 446
Rows covered with metadata expansion: 0
Rows with phase2/modeling enrichment: 446
Minimum modeling words: 92
Median modeling words: 126.0
final_enrichment_source_type
original_description_only          7402
wikipedia_full_extract_fallback     400
plot_or_synopsis_section             46


,tconst,primary_title,release_year,genres,final_enrichment_source_type,enrichment_word_count,modeling_description_word_count
0,tt0102926,The Silence of the Lambs,1991,"Crime,Drama,Thriller",plot_or_synopsis_section,733,959
1,tt0103064,Terminator 2: Judgment Day,1991,"Action,Sci-Fi",plot_or_synopsis_section,658,887
2,tt0110357,The Lion King,1994,"Adventure,Animation,Drama",plot_or_synopsis_section,663,878
3,tt0110912,Pulp Fiction,1994,"Crime,Drama",plot_or_synopsis_section,623,847
4,tt0111161,The Shawshank Redemption,1994,Drama,plot_or_synopsis_section,704,909
5,tt0120338,Titanic,1997,"Drama,Romance",original_description_only,0,135
6,tt0121164,Corpse Bride,2005,"Animation,Drama,Family",plot_or_synopsis_section,680,901
7,tt0172495,Gladiator,2000,"Action,Adventure,Drama",plot_or_synopsis_section,705,909
8,tt0212720,A.I. Artificial Intelligence,2001,"Drama,Sci-Fi",plot_or_synopsis_section,694,923
9,tt0217930,Egypt Uncovered,1998,Documentary,original_description_only,0,96


In [4]:
problem_titles = ["Inside Out", "Invincible", "The Bad Guys", "Demon Slayer: Kimetsu no Yaiba", "The Promised Neverland", "Home", "The Girl Who Leapt Through Time"]
cols = ["tconst", "content_type", "primary_title", "release_year", "genres", "final_enrichment_source_type", "enrichment_word_count", "modeling_description_word_count", "description", "enrichment_text"]
display(out[out["primary_title"].isin(problem_titles)][cols].sort_values(["primary_title", "release_year"]))


,tconst,content_type,primary_title,release_year,genres,final_enrichment_source_type,enrichment_word_count,modeling_description_word_count,description,enrichment_text
5881,tt9335498,tvSeries,Demon Slayer: Kimetsu no Yaiba,2019,"Action,Adventure,Animation",plot_or_synopsis_section,69,220,"Set in Taishō era Japan, the series follows Tanjiro Kamado , a teenage boy who joins the Demon Slayer Corps—an organization consisting of humans seeking to hunt down demons —after his family is slaughtered and his sister Nezuko is turne...","Set in Taishō era Japan, the series follows Tanjiro Kamado , a teenage boy who joins the Demon Slayer Corps—an organization consisting of humans seeking to hunt down demons —after his family is slaughtered and his sister Nezuko is turne..."
6870,tt1319569,movie,Home,2008,Drama,original_description_only,0,128,"Home is a 2008 Swiss drama film directed by Ursula Meier, starring Isabelle Huppert and Olivier Gourmet. The film was the official Swiss submission for Best Foreign Language Film at the 82nd Academy Awards, but was not nominated. Home i...",
1634,tt1014762,movie,Home,2009,Documentary,original_description_only,0,117,Home is a 2009 French documentary film by Yann Arthus-Bertrand. The film is almost entirely composed of aerial shots of various places on Earth. It shows the diversity of life on Earth and how humanity is threatening the ecological bala...,
3610,tt2224026,movie,Home,2015,"Adventure,Animation,Comedy",plot_or_synopsis_section,723,941,"A cowardly alien race called the Boov, led by Captain Smek, commences their ""friendly"" invasion on Earth. Relocating the humans, whom they deem as simple and backwards, to the Australian Outback , the Boov inhabit their homes. Oh, an ac...","A cowardly alien race called the Boov, led by Captain Smek, commences their ""friendly"" invasion on Earth. Relocating the humans, whom they deem as simple and backwards, to the Australian Outback , the Boov inhabit their homes. Oh, an ac..."
3514,tt2096673,movie,Inside Out,2015,"Adventure,Animation,Comedy",plot_or_synopsis_section,647,852,"In the mind of a young girl named Riley Andersen are five personified emotions that influence her actions: Joy , Sadness , Fear , Disgust , and Anger . They process her experiences into memories stored as colored orbs in ""Headquarters"",...","In the mind of a young girl named Riley Andersen are five personified emotions that influence her actions: Joy , Sadness , Fear , Disgust , and Anger . They process her experiences into memories stored as colored orbs in ""Headquarters"",..."
1209,tt0445990,movie,Invincible,2006,"Biography,Drama,Sport",original_description_only,0,137,"Invincible is a 2006 American biographical sports drama film directed by Ericson Core. It is based on the story of Vince Papale, a bartender who became a member of the Philadelphia Eagles in 1976 with the help of his coach, Dick Vermeil...",
5135,tt6741278,tvSeries,Invincible,2021,"Action,Adventure,Animation",plot_or_synopsis_section,95,196,"Mark Grayson is a seemingly typical teenager whose father, Nolan Grayson , also known as ""Omni-Man"", is the most powerful superhero on Earth. Shortly after his 17th birthday, Mark begins to develop powers of his own and learns how to wi...","Mark Grayson is a seemingly typical teenager whose father, Nolan Grayson , also known as ""Omni-Man"", is the most powerful superhero on Earth. Shortly after his 17th birthday, Mark begins to develop powers of his own and learns how to wi..."
5538,tt8115900,movie,The Bad Guys,2022,"Adventure,Animation,Comedy",plot_or_synopsis_section,691,920,"In Los Angeles, where humans and anthropomorphic animals co-exist, master pickpocketer Mr. Wolf leads the Bad Guys, an infamous gang of criminals consisting of his best friend and safecracker Mr. Snake, expert hacker Ms. ""Webs"" Tarantul...","In Los Angeles, where humans and anthropomorphic animals co-exist, master pickpocketer Mr. Wolf leads the Bad Guys, an infamous gang of criminals consisting of hi